# Grpo Reproduction

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installation

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} {get_pil} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

Load up `Llama 3.1 8B Instruct`, and set parameters

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


INFO 11-03 00:28:50 [__init__.py:244] Automatically detected platform cuda.
ERROR 11-03 00:28:53 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 11-03 00:29:02 [vllm_utils.py:694] Unsloth: Patching vLLM v1 graph capture
INFO 11-03 00:29:02 [vllm_utils.py:722] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.10.12: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit with actual 

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 11-03 00:29:23 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 11-03 00:29:23 [config.py:1472] Using max model len 1024
WARNING 11-03 00:29:23 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 11-03 00:29:26 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=2048.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.1.mlp'], 'llm_int8_threshold': 6.0}
INFO 11-03 00:29:26 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.2) with config: model='unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit', speculative_config=None, tok

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

INFO 11-03 00:31:41 [model_runner.py:1671] Graph capturing finished in 28 secs, took 0.59 GiB
INFO 11-03 00:31:41 [vllm_utils.py:734] Unsloth: Patched vLLM v0 graph capture finished in 28 secs.
INFO 11-03 00:31:42 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 40.62 seconds
Unsloth: Just some info: will skip parsing ['input_layernorm', 'pre_feedforward_layernorm', 'attention_norm', 'norm2', 'k_norm', 'post_attention_layernorm', 'post_layernorm', 'layer_norm2', 'post_feedforward_layernorm', 'norm1', 'layer_norm1', 'q_norm', 'ffn_norm']
Unsloth: Just some info: will skip parsing ['input_layernorm', 'cross_attn_post_attention_layernorm', 'pre_feedforward_layernorm', 'attention_norm', 'norm2', 'k_norm', 'post_attention_layernorm', 'post_layernorm', 'layer_norm2', 'post_feedforward_layernorm', 'norm1', 'cross_attn_input_layernorm', 'layer_norm1', 'q_norm', 'ffn_norm']


tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth 2025.10.12 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [ ]:
import re
from datasets import load_dataset, Dataset

# Load and prep dataset
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# uncomment middle messages for 1-shot prompting
def get_gsm8k_questions(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    }) # type: ignore
    return data # type: ignore

dataset = get_gsm8k_questions()

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [ ]:
max_prompt_length = 256

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 6, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_seq_length - max_prompt_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 250,
    save_steps = 250,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 6


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

OSError: Unsloth: LoRA filepath = grpo_saved_lora does not exist!

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

# Test

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# # Evaluate on GSM8K test split (Accuracy)

#
import re, math, time, itertools
from typing import List, Optional
from datasets import load_dataset
from tqdm import tqdm
from vllm import SamplingParams

# === Reuse or redefine helpers (in case the previous cell wasn't run) ===
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
""".strip()

def extract_xml_answer(text: str) -> Optional[str]:
    """Extract text inside <answer> ... </answer>. Return None if not found."""
    try:
        ans = text.split("<answer>")[-1]
        ans = ans.split("</answer>")[0]
        return ans.strip()
    except Exception:
        return None

def extract_hash_answer(text: str) -> Optional[str]:
    """GSM8K gold answers often end with '#### 42' — extract the part after ####."""
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

SOFT_XML_PATTERN = re.compile(r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>", re.DOTALL)

def normalize_num_like(s: str) -> str:
    """
    Normalize a numeric-like string for robust comparison:
    - strip spaces
    - drop trailing periods
    - remove commas
    - if it's a valid int after cleanup, cast to int then back to str
    """
    if s is None:
        return ""
    t = s.strip().rstrip(".").replace(",", "")
    # Remove currency/units commonly seen in GSM8K answers
    t = t.replace("$", "").replace("USD", "").replace("usd", "")
    # Some answers may contain extra text like 'The answer is 42'
    # Try to extract the last integer token if present
    tokens = re.findall(r"-?\d+", t)
    if len(tokens) == 1 and t == tokens[0]:
        # Pure integer string
        try:
            return str(int(tokens[0]))
        except Exception:
            return t
    elif len(tokens) >= 1:
        # Prefer the last number token as the final numeric answer
        try:
            return str(int(tokens[-1]))
        except Exception:
            pass
    # Fall back to plain cleaned string
    return t

def relaxed_equal(pred: str, gold: str) -> bool:
    """
    Relaxed comparison:
    - numeric-normalized equality if both look numeric-ish
    - else, strict string equality after strip
    """
    if gold is None or pred is None:
        return False
    p = normalize_num_like(pred)
    g = normalize_num_like(gold)
    # If both parse cleanly to int, compare as ints
    try:
        return int(p) == int(g)
    except Exception:
        pass
    # Otherwise compare stripped text
    return p.strip() == g.strip()

# === Config ===
LORA_PATH = "/content/drive/MyDrive/ColabNotebooks/GRPO/grpo_saved_lora"   # <- 如果你保存到其他目录，这里改路径
MAX_SAMPLES = None              # 设为整数以抽样测试，例如 200；None = 全量 test
BATCH_SIZE = 8                  # T4 推荐 4~16，按显存情况调整
TEMPERATURE = 0.2               # 评估更稳定，尽量低
TOP_P = 0.95
MAX_TOKENS = 512                # 生成长度上限，按你的 max_seq_length 决定
SEED = 3407


In [ ]:
# === Load test split (GSM8K or AIME) ===
from datasets import load_dataset
import re

DATASET = "aime2024"   # 可选: "gsm8k" 或 "aime2024" 或 "aime2025"

prompts, gold_answers = [], []

if DATASET == "gsm8k":
    raw_test = load_dataset("openai/gsm8k", "main")["test"]
    for x in raw_test:
        gold = extract_hash_answer(x["answer"])
        chat = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": x["question"]},
        ]
        prompts.append(chat)
        gold_answers.append(gold)

elif DATASET == "aime2024":
    ds = load_dataset("HuggingFaceH4/aime_2024")["train"]   # 30题
    for x in ds:
        gold = str(x["answer"]).strip()
        chat = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": x["problem"]},
        ]
        prompts.append(chat)
        gold_answers.append(gold)

elif DATASET == "aime2025":
    ds = load_dataset("opencompass/AIME2025")["train"]      # 30题
    for x in ds:
        gold = str(x["answer"]).strip()
        chat = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": x["problem"]},
        ]
        prompts.append(chat)
        gold_answers.append(gold)

else:
    raise ValueError("Unknown DATASET type!")

# 抽样（沿用你原来的变量）
if isinstance(MAX_SAMPLES, int) and MAX_SAMPLES > 0:
    prompts = prompts[:MAX_SAMPLES]
    gold_answers = gold_answers[:MAX_SAMPLES]

# === Build input texts via chat template ===
texts = [
    tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True)
    for p in prompts
]


README.md:   0%|          | 0.00/932 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/81.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30 [00:00<?, ? examples/s]

In [ ]:
# === Build input texts via chat template ===
# 依赖上个训练/推理单元里创建的 tokenizer 和 model 实例
# 如果你的 notebook 中 tokenizer/model 不在内存，请先按之前单元重新加载/构建。
texts = [
    tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True)
    for p in prompts
]

# === Prepare generation params & load LoRA once ===
sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
    seed=SEED,
)
lora_req = model.load_lora(LORA_PATH)

# === Run batched inference ===
def batched(iterable, n):
    it = iter(iterable)
    while True:
        chunk = list(itertools.islice(it, n))
        if not chunk:
            return
        yield chunk

start_time = time.time()
n_total = len(texts)
n_pred_ok = 0
n_xml_ok = 0
n_softformat_ok = 0
all_latencies = []
pred_samples = []  # store a few samples for inspection

with torch.no_grad():
    pbar = tqdm(total=n_total, desc="Evaluating (GSM8K test)")
    offset = 0
    for batch in batched(texts, BATCH_SIZE):
        t0 = time.time()
        outputs = model.fast_generate(
            batch,
            sampling_params=sampling_params,
            lora_request=lora_req,
        )
        latency = time.time() - t0
        all_latencies.append(latency / len(batch))

        for i, out in enumerate(outputs):
            gen_text = out.outputs[0].text  # vLLM single best output
            xml_ans = extract_xml_answer(gen_text)
            gold = gold_answers[offset + i]

            # metrics
            if xml_ans is not None:
                n_xml_ok += 1
            if SOFT_XML_PATTERN.search(gen_text) is not None:
                n_softformat_ok += 1
            if xml_ans is not None and relaxed_equal(xml_ans, gold):
                n_pred_ok += 1

            # keep a few examples for sanity check
            if len(pred_samples) < 5:
                pred_samples.append({
                    "gold": gold,
                    "extracted_answer": xml_ans,
                    "raw_output_tail": gen_text[-400:],  # last 400 chars
                })

        offset += len(batch)
        pbar.update(len(batch))
    pbar.close()

elapsed = time.time() - start_time

# === Report ===
acc = n_pred_ok / n_total if n_total else 0.0
xml_parse_rate = n_xml_ok / n_total if n_total else 0.0
soft_format_rate = n_softformat_ok / n_total if n_total else 0.0
avg_latency = sum(all_latencies) / len(all_latencies) if all_latencies else float("nan")

print(f"\n=== GSM8K Test Results ===")
print(f"Samples         : {n_total}")
print(f"Accuracy        : {acc:.4f}")
print(f"XML parse rate  : {xml_parse_rate:.4f}  (# with <answer>…</answer> extracted)")
print(f"Format (soft)   : {soft_format_rate:.4f}  (# matching <reasoning>…</reasoning><answer>…</answer>)")
print(f"Avg gen latency : {avg_latency:.3f} s / sample (batch_size={BATCH_SIZE})")
print(f"Wall time       : {elapsed:.1f} s")

print("\n--- A few prediction samples ---")
for k, s in enumerate(pred_samples, 1):
    print(f"\n[{k}] gold={s['gold']}")
    print(f"    pred={s['extracted_answer']}")
    print("    tail:", s["raw_output_tail"].replace("\n","\\n")[:400])


Evaluating (GSM8K test):   0%|          | 0/30 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Evaluating (GSM8K test):  27%|██▋       | 8/30 [01:51<05:05, 13.91s/it]

Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

...

Evaluating (GSM8K test):  80%|████████  | 24/30 [05:20<01:19, 13.29s/it]

Adding requests:   0%|          | 0/6 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/6 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Evaluating (GSM8K test): 100%|██████████| 30/30 [07:02<00:00, 14.08s/it]


=== GSM8K Test Results ===
Samples         : 30
Accuracy        : 0.0000
XML parse rate  : 1.0000  (# with <answer>…</answer> extracted)
Format (soft)   : 0.0333  (# matching <reasoning>…</reasoning><answer>…</answer>)
Avg gen latency : 14.259 s / sample (batch_size=8)
Wall time       : 422.3 s

--- A few prediction samples ---

[1] gold=204
    pred=Let's break down the information given:

1. At a speed of $s$ km/h, the total time taken is 4 hours, including $t$ minutes in the coffee shop. We can convert this to just hours by converting $t$ minutes to hours: $t$ minutes is $\frac{t}{60}$ hours. So the total time taken is $4 = \frac{9}{s} + \frac{t}{60}$.

2. At a speed of $s+2$ km/h, the total time taken is 2 hours and 24 minutes, including $t$ minutes in the coffee shop. Converting 24 minutes to hours gives us $\frac{24}{60} = \frac{2}{5}$ hours. So the total time taken is $2 + \frac{2}{5} = \frac{12}{5} = \frac{9}{s+2} + \frac{t}{60}$.

Now we have two equations:

$\frac{9}{s} + \f

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )